<div style="border-left:8px solid #BC8F8F;background:#ffffff;padding:14px 18px;border-radius:4px"><div style="font:600 13px sans-serif;color:#BC8F8F;letter-spacing:.08em">PART A</div><div style="font:600 28px sans-serif;margin-top:2px;color:#1f2430">Presentation</div><div style="font:15px sans-serif;margin-top:6px;color:#1f2430">Watch and follow along. Searching a collection by what images look like.</div></div>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 1 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

# Notebook 3: Searching by what images look like

The Europeana section of the intro notebook found objects through their *metadata*. This notebook searches by **content**: you describe an image in words, or give an image, and get the most similar ones from a shared collection.

It rests on an **embedding**: a list of numbers (here 512) that a neural network computes from an image, so that images with similar content get similar numbers. [CLIP](https://openai.com/research/clip) computes embeddings for images *and* short texts in the same space, so a sentence can be compared directly with every image.

In [ ]:
# Setup: run this cell first.
import numpy as np
from IPython.display import display
from clip_helpers import *
from iiif_helpers import show_mirador_many, url_panel

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 2 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## How the dataset was made

The dataset contains **no images**, only numbers and links. It was prepared once by the instructor:

```
Europeana Search API  →  metadata records (title, creator, institution, rights, links)
        │   for every record: request a small version of the image
        │      from its IIIF image service, or from Europeana's thumbnail service
        ▼
CLIP image encoder  →  one vector of 512 numbers per image   (the image is then discarded)
```

Only openly reusable images from Europeana's **Art** collection are included. The **originals and the small versions stay at the institutions**: when a search returns results, your computer fetches *those* images from their URLs, and nothing else.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 3 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## The collection

`metadata.csv` and the vectors are in the same order: row *i* describes vector *i*. A random sample (the images are fetched from their institutions now):

In [ ]:
coll = Collection()
coll.summary()

rng = np.random.default_rng(1)
coll.show_grid(rng.choice(len(coll.meta), 12, replace=False))

The Art collection is broader than paintings: it also holds drawings, prints, photographs, decorative objects, coins, maps and ethnographic items. Who contributes the most?

In [ ]:
coll.top_institutions_and_countries()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 4 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## CLIP: text and images in one space

![CLIP contrastive pre-training: a text encoder and an image encoder produce vectors that are compared in a matrix](../misc/assets/img/clip.svg)

<small>Contrastive pre-training of CLIP: matching image-text pairs are pulled together (blue diagonal), all other pairs pushed apart. Figure: OpenAI, ["CLIP: Connecting text and images"](https://openai.com/index/clip/) (Radford et al., 2021).</small>

The image half was applied to the whole collection beforehand. For a query only the **text encoder** runs, which is fast on a laptop.

In [ ]:
clip_model = Clip()

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 5 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Search with text

The query becomes a vector in the same 512-dimensional space. Both are normalised to length 1, so their **dot product** is their cosine similarity: 1 means the same direction, near 0 unrelated. Ranking the whole collection is one matrix–vector product.

In [ ]:
QUERY = "a ship on a stormy sea"

scores = coll.similarity(clip_model.text(QUERY))
best = coll.top_k(scores, 12)
coll.show_grid(best, scores)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 6 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Restricting the search with a mask

Because the metadata is aligned with the vectors, a filter is just a boolean **mask** applied before ranking. Here: only images dated before 1900.

In [ ]:
QUERY = "a portrait of a person"
mask = (coll.year > 0) & (coll.year < 1900)                          # dated, and before 1900
# mask = np.isin(coll.country, ["Sweden", "Finland", "Norway"])      # alternative: by country
# mask = np.isin(coll.institution, ["Wellcome Collection"])          # alternative: by institution

scores = coll.similarity(clip_model.text(QUERY))
print(f"{mask.sum():,} of {len(mask):,} images pass the filter. Top results for {QUERY!r}:\n")
coll.show_grid(coll.top_k(scores, 12, mask), scores)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 7 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Where do the results come from?

A result is only as varied as the collection it searches. Comparing the institutions in the top results with those in the whole collection shows whether a query favours some sources.

In [ ]:
QUERY = "a ship on a stormy sea"
scores = coll.similarity(clip_model.text(QUERY))
top = coll.top_k(scores, 100)
coll.institution_share(top, QUERY)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 8 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Search with an image

The same space holds image embeddings, so an image can be the query. Here: the best match of the ship query, used as "more like this".

In [ ]:
SEED = int(top[0])
print("Query: collection image", SEED, "-", coll.meta[SEED]["title"][:60])
display(coll.small_image(SEED))

scores = coll.similarity(coll.V[SEED].astype(np.float32))
scores[SEED] = -1                          # do not return the query itself
coll.show_grid(coll.top_k(scores, 12), scores)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>PART A · 9 / 9</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Look closely: a larger version, on request

The small versions are enough for CLIP, not for studying a detail. Once a result is interesting, a larger version or a detail can be requested from the institution:

- **With a IIIF image service**: one URL returns a detail at the size you ask for, and nothing else is downloaded.
- **Without one**: the whole original file has to be downloaded, often several megabytes.

First a result with an image service: the central quarter of the image at 1,000 pixels.

In [ ]:
QUERY = "a ship on a stormy sea"
scores = coll.similarity(clip_model.text(QUERY))
iiif_hits = coll.top_k(scores, 6, coll.has_iiif)

coll.show_grid(iiif_hits, scores)
coll.look_closely(int(iiif_hits[0]), size=1000, region=(0.25, 0.25, 0.5, 0.5))

Then the best match overall, which may have no image service. Compare the amount of data transferred:

In [ ]:
overall = int(coll.top_k(scores, 1)[0])
coll.look_closely(overall, size=1000)

<div style="border-left:8px solid #FF7F50;background:#ffffff;padding:14px 18px;border-radius:4px"><div style="font:600 13px sans-serif;color:#FF7F50;letter-spacing:.08em">PART B</div><div style="font:600 28px sans-serif;margin-top:2px;color:#1f2430">Hands-on</div><div style="font:15px sans-serif;margin-top:6px;color:#1f2430">Your turn: put it all together. About 15 minutes.</div></div>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>PART B · 1 / 2</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Exercise: build your own selection

Choose a **theme**, find matching images with CLIP, and look at your selection in one IIIF viewer. Then share it.

1. Pick a theme and write a **query** for it, such as `"a cat"`, `"a ship on a stormy sea"`, `"gold coins"`, `"a map of a city"` or `"a faded black and white photograph"`.
2. Optionally restrict it with a **mask** (period, country or institution).
3. Run the cell, then note the **numbers** of 3 to 5 results you like.

In [ ]:
# ============================================================
# EXERCISE: your theme
# ============================================================
QUERY = "a ship on a stormy sea"      # <-- your theme, described in words (English works best)
mask  = None                          # <-- or e.g. (coll.year > 0) & (coll.year < 1900)
                                      #     or np.isin(coll.country, ["Sweden", "Finland"])
# ============================================================

scores = coll.similarity(clip_model.text(QUERY))
best = coll.top_k(scores, 12, mask)
coll.show_grid(best, scores, numbered=True)

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#FF7F50"><span>PART B · 2 / 2</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #FF7F50;margin:6px 0 0 0">

## Exercise: from your picks to a viewer

Enter the numbers of the results you chose. The cell builds the **IIIF manifest URL** of each record (the pattern from the intro notebook), checks that it answers, and prints a block to paste into the shared pad.

In [ ]:
PICKS = [0, 1, 2]        # <-- the numbers of the results you chose, e.g. [0, 3, 5, 8]

chosen = [int(best[n]) for n in PICKS]
urls = [manifest_url(coll.meta[i]["id"]) for i in chosen]

print("Checking the manifests ...")
for i, u in zip(chosen, urls):
    ok = requests.get(u, timeout=30).ok
    print(("✓" if ok else "✗"), coll.meta[i]["title"][:60], "|", coll.meta[i]["institution"][:30])

print("\n----- paste into the shared pad -----")
print("Theme:", QUERY)
for u in urls:
    print(u)

In [ ]:
show_mirador_many(urls)

### Stretch

- Pick one result that has an image service and look at a **detail**: `coll.look_closely(chosen[0], region=(0.25, 0.25, 0.5, 0.5))`. Change the region (fractions `x, y, width, height`) to frame something you want to point out.
- Search with **your own image**: see the Appendix (section C).
- Narrow the theme with a **mask**, for example only images from one country, and compare the results with the unmasked search.

<details><summary><b>Hints</b> (click to open)</summary>

- The numbers (`#0`, `#1`, ...) are printed on each card. They refer to the list of 12 results of the cell above, not to the collection.
- A ✗ means the record has no manifest. Pick another result and run the cell again.
- Descriptions work better than single words: `"an oil painting of flowers in a vase"` instead of `"flowers"`.
- If the viewer stays empty, check that the manifest URLs above start with `https://iiif.europeana.eu/presentation/`.
</details>

<div style="border-left:8px solid #C0C0C0;background:#ffffff;padding:14px 18px;border-radius:4px"><div style="font:600 13px sans-serif;color:#C0C0C0;letter-spacing:.08em">APPENDIX</div><div style="font:600 28px sans-serif;margin-top:2px;color:#1f2430">Background</div><div style="font:15px sans-serif;margin-top:6px;color:#1f2430">Details for reading afterwards. Not part of the presentation.</div></div>

## A. The dataset folder

| Path | Content |
|---|---|
| `metadata.csv` | one row per image: title, creator, year, institution, country, rights, the URL of the small version (`small_image_url`), the original file (`full_image_url`), and the IIIF image service when one answered |
| `embeddings/clip_vit_b32.npz` | the CLIP vectors, in the same order as `metadata.csv` |
| `models/` | the CLIP model weights |

For every record the small version was requested from the IIIF image service (448 px) when the media URL is one, and otherwise from Europeana's thumbnail service (400 px wide). A small version is all CLIP needs: it resizes every image to 224 × 224 pixels anyway. The helper functions are in `clip_helpers.py`.

## B. Writing queries

| Instead of | Try |
|---|---|
| `water` | `a river flowing through a landscape` |
| `old` | `a faded black and white photograph` |
| `art` | `an oil painting of flowers in a vase` |

CLIP understands descriptions better than single keywords, and it works best in English. It also matches *style and medium* ("a pencil drawing", "a photograph") as well as content, which is useful for a mixed collection.

## C. Search with your own image

Put an image file next to the notebook and give its name. Its embedding is computed with the image half of CLIP, on your computer.

In [ ]:
YOUR_IMAGE = None            # <-- e.g. "my_photo.jpg" (a file next to the notebook)

if YOUR_IMAGE:
    from PIL import Image
    query_image = Image.open(YOUR_IMAGE)
    display(query_image.copy().resize((200, int(200 * query_image.height / query_image.width))))
    scores = coll.similarity(clip_model.image(query_image))
    coll.show_grid(coll.top_k(scores, 12), scores)
else:
    print("Set YOUR_IMAGE to the name of an image file and run again.")

## D. Summary

- An **embedding** turns an image or a text into a vector. CLIP puts both in one space, so a sentence can be compared with images and an image with images.
- Searching a precomputed collection is a **matrix–vector product**, plus optional masks built from the metadata.
- The collection is a sample that reflects what institutions made available. Results show *what is in the collection*, so it pays to check where they come from.
- Small images are enough to compute embeddings. A larger version or a detail is requested **only when needed**, through the IIIF image API where the institution offers one, and as a full download where not.

### Limits to keep in mind

- CLIP was trained on images and captions from the web. It reflects the language, subjects and biases of that data, and works best on English queries and on photographic or pictorial content.
- Similarity is not meaning. Two images can be close because of composition, colour or medium and not because of subject.
- The dataset is a selection (openly reusable images from Europeana's Art collection), not Europeana as a whole. The Art collection is Europeana's own curation and is broader than fine art.
- The images live at the institutions. If a source is offline or has moved the file, that image shows as a grey square, and the links in `metadata.csv` reflect the time the dataset was built.
- Detecting IIIF services from the media URL finds many but not all of them. Some institutions have a service that cannot be seen from the URL, and are served through Europeana's thumbnails.